In [1]:
import sys
import json

sys.path.append("../src")

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.utils.function_calling import convert_to_openai_tool
from mortgage_agent.retrieval import retrieve
from mortgage_agent.tools import ltv, dti, funds_to_close, reserves_months, add_years, search_guide
from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
print(type(ltv))
print(ltv.name)
print(ltv.description)
print(ltv.args)
print(ltv.invoke({"loan_amount": 320000, "property_value": 400000}))

<class 'langchain_core.tools.structured.StructuredTool'>
ltv
Loan-to-value ratio as a percentage: loan amount divided by property value, times 100.
{'loan_amount': {'title': 'Loan Amount', 'type': 'number'}, 'property_value': {'title': 'Property Value', 'type': 'number'}}
80.0


In [3]:
print(json.dumps(convert_to_openai_tool(ltv), indent=2))

{
  "type": "function",
  "function": {
    "name": "ltv",
    "description": "Loan-to-value ratio as a percentage: loan amount divided by property value, times 100.",
    "parameters": {
      "properties": {
        "loan_amount": {
          "type": "number"
        },
        "property_value": {
          "type": "number"
        }
      },
      "required": [
        "loan_amount",
        "property_value"
      ],
      "type": "object"
    }
  }
}


In [4]:
print(dti.args)
print(dti.invoke({"housing_payment": 2420, "monthly_debts": [450, 120], "monthly_income": 8500}))
print(funds_to_close.invoke({"property_value": 400000, "loan_amount": 320000, "closing_costs": 9000}))
print(reserves_months.invoke({"available_funds": 55000, "cash_to_close": 89000, "monthly_pitia": 2420}))
print(reserves_months.invoke({"available_funds": 95000, "cash_to_close": 89000, "monthly_pitia": 2420}))
print(add_years.invoke({"start_date": "2022-10-30", "years": 4}))
print(add_years.invoke({"start_date": "2024-02-29", "years": 1}))

{'housing_payment': {'title': 'Housing Payment', 'type': 'number'}, 'monthly_debts': {'items': {'type': 'number'}, 'title': 'Monthly Debts', 'type': 'array'}, 'monthly_income': {'title': 'Monthly Income', 'type': 'number'}}
35.18
89000.0
-14.05
2.48
2026-10-30
2025-02-28


In [5]:
from mortgage_agent.retrieval import retrieve

for q in ["large deposit documentation", "Chapter 7 bankruptcy waiting period", "gift from parents"]:
    print(q, "→", [d.metadata["rule_id"] for d in retrieve(q, top_n=3)])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

large deposit documentation → ['B3-4.2-02', 'B3-4.2-02', 'B3-4.2-02']
Chapter 7 bankruptcy waiting period → ['B3-5.3-07', 'B3-5.3-07', 'B3-5.3-07']
gift from parents → ['B3-4.3-04', 'B3-4.3-04', 'B3-4.3-05']


In [6]:
result = search_guide.invoke({"query": "large deposit documentation"})
print(len(result), "characters")
print(result[:700])

3797 characters
Fannie Mae Selling Guide, B3-4.2-02: Depository Accounts

loan.  When bank statements (typically covering the most recent two months) are used, the lender must
evaluate large deposits. See B3-4.1-04, Virtual Currency for additional information when a large deposit may be
from virtual currency that was exchanged into U.S. dollars. Requirements for evaluating large deposits vary
based on the transaction type, as shown in the table below.
Transaction Type
Evaluation Requirements
Refinance transactions
Documentation or explanation for large deposits is not required; however,
the lender remains responsible for ensuring that any borrowed funds,
including any related liability, are considered.
Tran


In [7]:
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
tools = [search_guide, ltv, dti, funds_to_close, reserves_months, add_years]
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What is the LTV for a $320,000 loan on a $400,000 home?")
print("Text:", repr(response.content))
print("Tool calls:", response.tool_calls)
print(response.tool_calls[0])

Text: ''
Tool calls: [{'name': 'ltv', 'args': {'loan_amount': 320000, 'property_value': 400000}, 'id': 'fc_bae67d3b-7d20-4b6f-892d-30ad4225be80', 'type': 'tool_call'}]
{'name': 'ltv', 'args': {'loan_amount': 320000, 'property_value': 400000}, 'id': 'fc_bae67d3b-7d20-4b6f-892d-30ad4225be80', 'type': 'tool_call'}


In [8]:
call = response.tool_calls[0]
print(ltv.invoke(call["args"]))

80.0


In [9]:
response = llm_with_tools.invoke("What does LTV stand for?")
print("Text:", response.content)
print("Tool calls:", response.tool_calls)

Text: **LTV** stands for **Loan‑to‑Value** ratio.  

It’s a percentage that compares the amount of the mortgage loan to the appraised (or purchase) value of the property:

\[
\text{LTV} = \frac{\text{Loan Amount}}{\text{Property Value}} \times 100\%
\]

Lenders use the LTV to assess risk—the higher the LTV, the less equity the borrower has in the home, which generally means a higher risk for the lender. Consequently, higher‑LTV loans often require mortgage insurance or come with higher interest rates.
Tool calls: []


In [10]:
response = llm_with_tools.invoke("Does a $40,000 bank deposit need to be documented for a mortgage?")
print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'search_guide', 'args': {'query': 'large deposit documentation'}, 'id': 'fc_9e1cdcc7-0301-4c79-8e41-ba20f5283c07', 'type': 'tool_call'}]


In [11]:
from langchain_core.messages import HumanMessage, ToolMessage

tools_by_name = {t.name: t for t in tools}


def run_agent(question: str, max_steps: int = 10) -> str:
    """Ask the LLM, run any tools it needs, and repeat until it answers."""
    messages = [HumanMessage(question)]
    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            print(f"Step {step + 1}: {call['name']}({call['args']})")
            result = tools_by_name[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "Stopped: too many steps."
        

In [12]:
print(run_agent("What is the LTV for a $320,000 loan on a $400,000 home?"))

Step 1: ltv({'loan_amount': 320000, 'property_value': 400000})
The loan‑to‑value (LTV) ratio is calculated as:

\[
\text{LTV} = \frac{\text{Loan Amount}}{\text{Property Value}} \times 100
\]

For a $320,000 loan on a $400,000 home:

\[
\text{LTV} = \frac{320{,}000}{400{,}000} \times 100 = 0.80 \times 100 = 80\%
\]

**LTV = 80 %**.


In [13]:
print(run_agent(
    "A borrower's Chapter 7 bankruptcy was discharged on 2022-10-31. "
    "The new loan closes on 2026-10-30. Is the waiting period met?"
))

Step 1: search_guide({'query': 'Chapter 7 bankruptcy waiting period'})
**Short answer:** No – the standard 4‑year waiting period for a Chapter 7 bankruptcy is not yet satisfied. The loan closes one day before the required date.

**Why**

| Item | Requirement (Fannie Mae Selling Guide B3‑5.3‑07) | Your dates |
|------|-----------------------------------------------|------------|
| **Bankruptcy type** | Chapter 7 (or Chapter 11) | – |
| **Standard waiting period** | **4 years** measured from the **discharge date** | Discharge = 2022‑10‑31 → 4‑year mark = **2026‑10‑31** |
| **Extenuating‑circumstances waiting period** | 2 years (if documented extenuating circumstances) | 2‑year mark = 2024‑10‑31 (already passed) |
| **Loan closing date** | 2026‑10‑30 | – |

- The loan is scheduled to close on **2026‑10‑30**, which is **3 years, 364 days** after the discharge.  
- The required 4‑year period ends on **2026‑10‑31**. Because the closing is **one calendar day early**, the waiting period is **n

In [ ]:
# 1. LLM + tools
llm_with_tools = llm.bind_tools(tools)

# Node: LLM
def call_llm(state: MessagesState) -> dict:
    """Node: send the whole conversation to the LLM and return its reply."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# 2. State  +  3. Nodes
builder = StateGraph(MessagesState)
builder.add_node("llm", call_llm)
builder.add_node("tools", ToolNode(tools))

# 4. Edges
builder.add_edge(START, "llm")                           # always start by thinking
builder.add_conditional_edges("llm", tools_condition)    # tools, or finish?
builder.add_edge("tools", "llm")                         # after acting, think again

# 5. Compile + run
graph = builder.compile()